<h2> What is ground-truth data or gold-standard data?</h2>

A dataset that acts as the benchmark for all evaluations.

In [1]:
# fetching the raw documents

import requests

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

# flatenning the raw documents' nested structure. run 'documents_raw' in a new cell to check the structure. then delete before saving. it is very big.

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [2]:
# checking the flattened structure now after adding course name to each and every document by de-nesting it.

documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [3]:
# creating a unique ID for each document using python's MD5 to create fixed 32-character hexadecimal hash from the input string

import hashlib

def generate_document_id(doc):
    combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
    hash_object = hashlib.md5(combined.encode())  # encode converts into bytes, and passes it to md5 to produce a fixed 32-char hexadecimal hash
    hash_hex = hash_object.hexdigest() # converts raw bytes into readable and storable hex string
    document_id = hash_hex[:8] # raw bytes couldn't be sliced to make it short
    return document_id

<h3> Why MD5 instead of UUID for creating ID?</h3>

1. MD5 creates the same ID for the same input. UUID is random. if you reprocess the data, the same doc gets the same ID. 
2. MD5 regenerates the same ID without storing it in a DB. UUID has to be store since it cannot be regenerated from the document later.
3. MD5's ID is stable as long as the documents' fields stay the same (course + question + text). UUID would break this, because every run would assign new IDs.

In [4]:
for doc in documents:
    doc['id'] = generate_document_id(doc)

documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp',
 'id': 'c02e79ef'}

In [5]:
# grouping documents by their id and then comparing how many unique IDs are there versus total documents

from collections import defaultdict

# creates a dictionary where every new key starts with an empty list automatically. each key = doc_id, each value = list of docs with that ID
hashes = defaultdict(list) 

for doc in documents:
    doc_id = doc['id']
    hashes[doc_id].append(doc)

# If doc_id is new, hashes[doc_id] automatically becomes an empty list [].
# Then .append(doc) adds the document to that list.
# This makes grouping items by key very clean.

In [6]:
print("no. of unique IDs = ", len(hashes))
print("no. of docuements = ", len(documents))

no. of unique IDs =  947
no. of docuements =  948


In [7]:
# finding the IDs that are being used for multiple documents

for k, values in hashes.items():
    if len(values) > 1:
        print(f"ID = {k}, no. of documents = {len(values)}")

# hashes.items() returns a python's view object(class = dict_items) of the hashes dict that shows all the key-value pairs. each item is a tuple of (key, value).
# k, values in hashes.items() is unpacking that tuple where k = key and values = value based on the positional order of writing.

ID = 593f7569, no. of documents = 2


In [8]:
# calling the document by the ID in the hashes dict

hashes['593f7569']

[{'text': "They both do the same, it's just less typing from the script.\nAsked by Andrew Katoch, Added by Edidiong Esu",
  'section': '6. Decision Trees and Ensemble Learning',
  'question': 'Does it matter if we let the Python file create the server or if we run gunicorn directly?',
  'course': 'machine-learning-zoomcamp',
  'id': '593f7569'},
 {'text': "They both do the same, it's just less typing from the script.",
  'section': '6. Decision Trees and Ensemble Learning',
  'question': 'Does it matter if we let the Python file create the server or if we run gunicorn directly?',
  'course': 'machine-learning-zoomcamp',
  'id': '593f7569'}]

In [9]:
# creating a JSON file to store the 'documents' in JSON format

import json
with open('documents-with-ids.json', 'wt') as f_out: # w - writing, t - text mode (as opposed to binary)
    json.dump(documents, f_out, indent = 2)

In [10]:
!head documents-with-ids.json

[
  {
    "text": "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  \u201cOffice Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon\u2019t forget to register in DataTalks.Club's Slack and join the channel.",
    "section": "General course-related questions",
    "question": "Course - When will the course start?",
    "course": "data-engineering-zoomcamp",
    "id": "c02e79ef"
  },
  {
    "text": "GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites",


In [11]:
# creating the prompt for gemini API

prompt_template = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record. 

The record:

section: {section}
question: {question}
answer: {text}

Provide the output in parsable JSON without using code blocks:

["question1", "question2", ..., "question5"]
""".strip()

In [18]:
# calling the gemini LLM API

import google.generativeai as genai

import os
api_key = os.environ.get("GOOGLE_API_KEY")

genai.configure(api_key = api_key)

model = genai.GenerativeModel('gemini-1.5-flash-latest')

In [19]:
# defining the function to create questions from the documents

def generate_questions(doc):
    prompt = prompt_template.format(**doc) # **doc unpacks the dictionary into keyword arguments
    json_response = model.generate_content(prompt)
    return json_response.text
    

In [23]:
from tqdm.auto import tqdm
# shows a progress bar—great for tracking long-running tasks.

import time
# for limiting the requests to gemini API due to the restrictions of a free tier plan.

In [26]:
results = {}

for doc in tqdm(documents):
    doc_id = doc['id']
    if doc_id in results:
        continue

    while True:
        try:
            questions = generate_questions(doc)
            results[doc_id] = questions
            time.sleep(5)  # Adds a 4s delay between requests to stay under the quota ~15 requests per minute.
            break
        except Exception as e:
            print(f"Rate limit exceeded. Retrying in 36 seconds...")
            time.sleep(36)

  0%|          | 0/948 [00:00<?, ?it/s]

Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...
Rate limit exceeded. Retrying in 36 seconds...


KeyboardInterrupt: 